# Random Forest Model:-

> Here is the complete BigQuery ML (BQML) pipeline for training a Random Forest Classification Model on the Telco-Customer-Churn.csv dataset to predict customer churn (Churn = 'Yes' or 'No').

# Key Things to Know About the Telco Churn Dataset:-
1. **The TotalCharges Bug**: In this dataset, rows where tenure = 0 often have blank spaces (" ") instead of numbers in TotalCharges. If left unhandled, BigQuery will import the whole column as a STRING or throw an error. We clean this safely using SAFE_CAST(TRIM(TotalCharges) AS FLOAT64).
2. **Exclude customerID**: Identifiers must be excluded so the model doesn't memorize customer IDs.
3. **Class Imbalance**: Roughly ~73% of customers stay and ~27% churn. We use auto_class_weights = TRUE so the Random Forest pays equal attention to churners.

# Complete Code:-

```sql
-- Create a schema/dataset in the US region
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);


-- Cleanse data: Fix TotalCharges and exclude customerID
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_telco_churn` AS
SELECT
  -- Target column (Yes / No)
  CAST(Churn AS STRING) AS Churn,
  
  -- Demographics
  CAST(gender AS STRING) AS gender,
  CAST(SeniorCitizen AS INT64) AS SeniorCitizen,
  CAST(Partner AS STRING) AS Partner,
  CAST(Dependents AS STRING) AS Dependents,
  
  -- Services Signed Up For
  CAST(tenure AS INT64) AS tenure,
  CAST(PhoneService AS STRING) AS PhoneService,
  CAST(MultipleLines AS STRING) AS MultipleLines,
  CAST(InternetService AS STRING) AS InternetService,
  CAST(OnlineSecurity AS STRING) AS OnlineSecurity,
  CAST(OnlineBackup AS STRING) AS OnlineBackup,
  CAST(DeviceProtection AS STRING) AS DeviceProtection,
  CAST(TechSupport AS STRING) AS TechSupport,
  CAST(StreamingTV AS STRING) AS StreamingTV,
  CAST(StreamingMovies AS STRING) AS StreamingMovies,
  
  -- Contract & Billing Information
  CAST(Contract AS STRING) AS Contract,
  CAST(PaperlessBilling AS STRING) AS PaperlessBilling,
  CAST(PaymentMethod AS STRING) AS PaymentMethod,
  CAST(MonthlyCharges AS FLOAT64) AS MonthlyCharges,
  
  -- Fix spaces in TotalCharges for new customers (tenure=0)
  COALESCE(SAFE_CAST(TRIM(CAST(TotalCharges AS STRING)) AS FLOAT64), 0.0) AS TotalCharges
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_telco_churn`;



-- Train the Random Forest Classifier
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`
OPTIONS (
  model_type = 'RANDOM_FOREST_CLASSIFIER',
  input_label_cols = ['Churn'],
  
  -- Random Forest Hyperparameters
  num_parallel_tree = 20,        -- Number of decision trees in the forest
  max_tree_depth = 6,            -- Maximum depth of each tree
  
  -- Split & Imbalance Handling
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2,
  auto_class_weights = TRUE      -- Balance churn vs non-churn weighting
) AS
SELECT
  *
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_telco_churn`;





-- Evaluate test performance
SELECT
  *
FROM
  ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`);




-- Confusion Matrix
SELECT
  *
FROM
  ML.CONFUSION_MATRIX(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`);




-- Check which features drive customer churn the most
SELECT
  feature,
  importance_gain,
  importance_weight
FROM
  ML.FEATURE_IMPORTANCE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`)
ORDER BY
  importance_gain DESC;



-- Predict churn probabilities on new customer profiles
SELECT
  predicted_Churn,
  predicted_Churn_probs,
  Contract,
  tenure,
  MonthlyCharges,
  InternetService
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`,
    (
      -- Customer 1: High Risk Profile
      SELECT
        CAST('Female' AS STRING) AS gender,
        CAST(0 AS INT64) AS SeniorCitizen,
        CAST('No' AS STRING) AS Partner,
        CAST('No' AS STRING) AS Dependents,
        CAST(2 AS INT64) AS tenure,
        CAST('Yes' AS STRING) AS PhoneService,
        CAST('No' AS STRING) AS MultipleLines,
        CAST('Fiber optic' AS STRING) AS InternetService,
        CAST('No' AS STRING) AS OnlineSecurity,
        CAST('No' AS STRING) AS OnlineBackup,
        CAST('No' AS STRING) AS DeviceProtection,
        CAST('No' AS STRING) AS TechSupport,
        CAST('Yes' AS STRING) AS StreamingTV,
        CAST('Yes' AS STRING) AS StreamingMovies,
        CAST('Month-to-month' AS STRING) AS Contract,
        CAST('Yes' AS STRING) AS PaperlessBilling,
        CAST('Electronic check' AS STRING) AS PaymentMethod,
        CAST(89.85 AS FLOAT64) AS MonthlyCharges,
        CAST(179.70 AS FLOAT64) AS TotalCharges
      UNION ALL
      -- Customer 2: Low Risk Profile
      SELECT
        CAST('Male' AS STRING) AS gender,
        CAST(0 AS INT64) AS SeniorCitizen,
        CAST('Yes' AS STRING) AS Partner,
        CAST('Yes' AS STRING) AS Dependents,
        CAST(60 AS INT64) AS tenure,
        CAST('Yes' AS STRING) AS PhoneService,
        CAST('Yes' AS STRING) AS MultipleLines,
        CAST('DSL' AS STRING) AS InternetService,
        CAST('Yes' AS STRING) AS OnlineSecurity,
        CAST('Yes' AS STRING) AS OnlineBackup,
        CAST('Yes' AS STRING) AS DeviceProtection,
        CAST('Yes' AS STRING) AS TechSupport,
        CAST('No' AS STRING) AS StreamingTV,
        CAST('No' AS STRING) AS StreamingMovies,
        CAST('Two year' AS STRING) AS Contract,
        CAST('No' AS STRING) AS PaperlessBilling,
        CAST('Bank transfer (automatic)' AS STRING) AS PaymentMethod,
        CAST(45.00 AS FLOAT64) AS MonthlyCharges,
        CAST(2700.00 AS FLOAT64) AS TotalCharges
    )
  );




-- Cleanup resources
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_telco_churn`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_telco_churn`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;

```

# Complete Step by step Guide for Random Forest Model:-

- **Step 1**: Create Schema / Dataset
```sql 
-- Create a schema/dataset in the US region
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);
```

> gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_telco_churn

---

- **Step 2**: Cleanse and Preprocess Data -> Fix TotalCharges, exclude customerID, and ensure correct data types.

```sql 
-- Cleanse data: Fix TotalCharges and exclude customerID
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_telco_churn` AS
SELECT
  -- Target column (Yes / No)
  CAST(Churn AS STRING) AS Churn,
  
  -- Demographics
  CAST(gender AS STRING) AS gender,
  CAST(SeniorCitizen AS INT64) AS SeniorCitizen,
  CAST(Partner AS STRING) AS Partner,
  CAST(Dependents AS STRING) AS Dependents,
  
  -- Services Signed Up For
  CAST(tenure AS INT64) AS tenure,
  CAST(PhoneService AS STRING) AS PhoneService,
  CAST(MultipleLines AS STRING) AS MultipleLines,
  CAST(InternetService AS STRING) AS InternetService,
  CAST(OnlineSecurity AS STRING) AS OnlineSecurity,
  CAST(OnlineBackup AS STRING) AS OnlineBackup,
  CAST(DeviceProtection AS STRING) AS DeviceProtection,
  CAST(TechSupport AS STRING) AS TechSupport,
  CAST(StreamingTV AS STRING) AS StreamingTV,
  CAST(StreamingMovies AS STRING) AS StreamingMovies,
  
  -- Contract & Billing Information
  CAST(Contract AS STRING) AS Contract,
  CAST(PaperlessBilling AS STRING) AS PaperlessBilling,
  CAST(PaymentMethod AS STRING) AS PaymentMethod,
  CAST(MonthlyCharges AS FLOAT64) AS MonthlyCharges,
  
  -- Fix spaces in TotalCharges for new customers (tenure=0)
  COALESCE(SAFE_CAST(TRIM(CAST(TotalCharges AS STRING)) AS FLOAT64), 0.0) AS TotalCharges
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_telco_churn`;
```

---

- **Step 3**: Train the Random Forest Model -> We train an ensemble of 20 trees (num_parallel_tree = 20) with a maximum depth of 6 per tree.

```sql 
-- Train the Random Forest Classifier
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`
OPTIONS (
  model_type = 'RANDOM_FOREST_CLASSIFIER',
  input_label_cols = ['Churn'],
  
  -- Random Forest Hyperparameters
  num_parallel_tree = 20,        -- Number of decision trees in the forest
  max_tree_depth = 6,            -- Maximum depth of each tree
  
  -- Split & Imbalance Handling
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2,
  auto_class_weights = TRUE      -- Balance churn vs non-churn weighting
) AS
SELECT
  *
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_telco_churn`;
```

---

- **Step 4**: Evaluate Model Performance -> Returns Accuracy, Precision, Recall, F1-Score, and ROC AUC.

```sql 
-- Evaluate test performance
SELECT
  *
FROM
  ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`);
```

---


- **Step 5**: Confusion Matrix -> See where the model makes False Alarms (predicting churn when they stay) vs Misses (predicting stay when they churn).

```sql 
-- Confusion Matrix
SELECT
  *
FROM
  ML.CONFUSION_MATRIX(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`);
```

---


- **Step 6**: Feature Importance -> Identify the top business drivers of churn (e.g., Contract, tenure, MonthlyCharges, InternetService).

```sql 
-- Check which features drive customer churn the most
SELECT
  feature,
  importance_gain,
  importance_weight
FROM
  ML.FEATURE_IMPORTANCE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`)
ORDER BY
  importance_gain DESC;
```

---


- **Step 7**: Predict Churn on New Customers -> Predict churn for two customer profiles:
    - **Customer 1 (High Churn Risk)**: Month-to-month contract, Fiber optic internet, high monthly charges, low tenure.
    - **Customer 2 (Low Churn Risk)**: Two-year contract, DSL internet, long tenure, automated payment.

```sql 
-- Predict churn probabilities on new customer profiles
SELECT
  predicted_Churn,
  predicted_Churn_probs,
  Contract,
  tenure,
  MonthlyCharges,
  InternetService
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`,
    (
      -- Customer 1: High Risk Profile
      SELECT
        CAST('Female' AS STRING) AS gender,
        CAST(0 AS INT64) AS SeniorCitizen,
        CAST('No' AS STRING) AS Partner,
        CAST('No' AS STRING) AS Dependents,
        CAST(2 AS INT64) AS tenure,
        CAST('Yes' AS STRING) AS PhoneService,
        CAST('No' AS STRING) AS MultipleLines,
        CAST('Fiber optic' AS STRING) AS InternetService,
        CAST('No' AS STRING) AS OnlineSecurity,
        CAST('No' AS STRING) AS OnlineBackup,
        CAST('No' AS STRING) AS DeviceProtection,
        CAST('No' AS STRING) AS TechSupport,
        CAST('Yes' AS STRING) AS StreamingTV,
        CAST('Yes' AS STRING) AS StreamingMovies,
        CAST('Month-to-month' AS STRING) AS Contract,
        CAST('Yes' AS STRING) AS PaperlessBilling,
        CAST('Electronic check' AS STRING) AS PaymentMethod,
        CAST(89.85 AS FLOAT64) AS MonthlyCharges,
        CAST(179.70 AS FLOAT64) AS TotalCharges
      UNION ALL
      -- Customer 2: Low Risk Profile
      SELECT
        CAST('Male' AS STRING) AS gender,
        CAST(0 AS INT64) AS SeniorCitizen,
        CAST('Yes' AS STRING) AS Partner,
        CAST('Yes' AS STRING) AS Dependents,
        CAST(60 AS INT64) AS tenure,
        CAST('Yes' AS STRING) AS PhoneService,
        CAST('Yes' AS STRING) AS MultipleLines,
        CAST('DSL' AS STRING) AS InternetService,
        CAST('Yes' AS STRING) AS OnlineSecurity,
        CAST('Yes' AS STRING) AS OnlineBackup,
        CAST('Yes' AS STRING) AS DeviceProtection,
        CAST('Yes' AS STRING) AS TechSupport,
        CAST('No' AS STRING) AS StreamingTV,
        CAST('No' AS STRING) AS StreamingMovies,
        CAST('Two year' AS STRING) AS Contract,
        CAST('No' AS STRING) AS PaperlessBilling,
        CAST('Bank transfer (automatic)' AS STRING) AS PaymentMethod,
        CAST(45.00 AS FLOAT64) AS MonthlyCharges,
        CAST(2700.00 AS FLOAT64) AS TotalCharges
    )
  );
```

---


- **Step 8**: Clean Up Resources (Optional)

```sql 
-- Cleanup resources
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.telco_churn_rf_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_telco_churn`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_telco_churn`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;
```

---
